# Lunar-HFE, part 1: run the analysis

This notebook reruns the analysis behind the JGR: Planets letter *"Difference of
the Lunar Regolith Thermal Conductivity $K_d$ at the Apollo 15 and 17 Heat-Flow
Boreholes from Global Model Values"* and its Supporting Information (SI), code
release **v2.3-jgr**. It writes every result to `code/results/`. The other two
notebooks check those results against the paper:

| Notebook | What it does |
|---|---|
| `1_run_analysis.ipynb` (this one) | checks the setup, then runs every analysis step |
| `2_letter_results.ipynb` | Table 1, Figures 3 and 4, and every number of the letter |
| `3_supporting_information.ipynb` | every number of the 14 SI tables, and SI Figures S5–S7 |

The repository ships every result, so notebooks 2 and 3 also work without
running this one.

**How to use it:** choose `MODE` in the first code cell below, then
**Run All** (JupyterLab: *Run ▸ Run All Cells*; VS Code: *Run All*).

| `MODE` | What runs | Time |
|---|---|---|
| `"show"` (default) | nothing is recomputed: every step reports its stored result in `code/results/` | a minute |
| `"quick"` | the short steps run; the long ones run their fast form (re-analysing stored model runs) or keep their stored result | about 45 min |
| `"full"` | every step from scratch, as for the paper | about 6 h on a six-core laptop |

Each step is one script in `code/pipeline/`. The cell runs it and shows its
printout as it goes, so this notebook and `make analysis` run exactly the same
code. On a Mac every step keeps the computer awake (`caffeinate`), but not with
the lid closed: keep it open during a long run.

In [1]:
# How much to run:  "show"  (minutes) | "quick"  (about 45 min) | "full"  (about 6 h)
MODE = "show"

try:
    from lunar import notebook as nb
except ImportError:
    raise ImportError("The lunar package is not installed. In a terminal, in the repository folder, run:\n"
                      "    pip install -e \".[dev,mcmc]\"\nthen restart the kernel.")
import numpy as np
import pandas as pd
from IPython.display import display
nb.set_mode(MODE)
S2 = ("A15", "A17")
DIVINER = ("data/diviner/gcp/global_cumul_avg_cyl_20n30n_002.tab", "data/diviner/gcp/global_cumul_avg_cyl_10n20n_002.tab")

Run mode: 'show'. Nothing is recomputed; every step shows its stored result.


---
# 1. Setup

## 1.1 Python and the core libraries
Python 3.10 or newer. A version problem is the most common reason a notebook
will not run, so it is checked first.

In [2]:
import sys, platform
print(f"Python {sys.version.split()[0]} on {platform.system()} {platform.machine()}")
assert sys.version_info >= (3, 10), "Python 3.10+ required"
import scipy, matplotlib, numba
for mod in (np, scipy, matplotlib, pd, numba):
    print(f"  {mod.__name__:<12} {mod.__version__}")

Python 3.13.13 on Darwin arm64


  numpy        2.4.6
  scipy        1.18.0
  matplotlib   3.11.1
  pandas       3.0.5
  numba        0.66.0


## 1.2 The `lunar` package and its constants
The physics is the installed package `lunar` (`code/src/lunar/`). This checks
the constants the study is defined by: the Stefan–Boltzmann constant and the
Hayne et al. (2017) regolith parameters.

In [3]:
import math
from lunar import constants as c
assert math.isclose(c.SIGMA_SB, 5.670374419e-8, rel_tol=1e-6), "SIGMA_SB constant wrong"
assert math.isclose(c.K_DEEP, 3.4e-3, rel_tol=1e-6), "Hayne K_d default is wrong"
assert math.isclose(c.K_SURFACE, 7.4e-4, rel_tol=1e-6), "Hayne K_s default is wrong"
assert math.isclose(c.H_PARAMETER, 0.06, rel_tol=1e-6), "Hayne H is wrong"
print(f"  sigma = {c.SIGMA_SB} W m^-2 K^-4")
print(f"  Hayne global K_d = {c.K_DEEP*1e3:.1f} mW m^-1 K^-1,  K_s = {c.K_SURFACE*1e3:.2f},  H = {c.H_PARAMETER*100:.0f} cm")

  sigma = 5.670374419e-08 W m^-2 K^-4
  Hayne global K_d = 3.4 mW m^-1 K^-1,  K_s = 0.74,  H = 6 cm


## 1.3 The Apollo heat-flow record
The 1971–1977 record (St. Clair et al., 2020; Nagihara et al., 2018) is bundled
in `code/data/apollo/`. Only the sensors below 80 cm are fitted: 7 at Apollo 15
and 16 at Apollo 17.

In [4]:
from lunar.apollo_helpers import extract_sensor_stability
for mission, expected in (("a15", 7), ("a17", 16)):
    n_deep = int((extract_sensor_stability(mission, min_depth_cm=80)["depth_cm_all"] >= 80).sum())
    print(f"  Apollo {mission[-2:]}: {n_deep} sensors below 80 cm (expected {expected})")
    assert n_deep == expected, f"sensor count mismatch for {mission}"

  Apollo 15: 7 sensors below 80 cm (expected 7)


  Apollo 17: 16 sensors below 80 cm (expected 16)


## 1.4 Solar geometry (SPICE)
The real-forcing run of the annual wave needs the Sun's position from the
bundled JPL DE440 kernels.

In [5]:
from lunar.ephem import et_from_iso, solar_vector_moon_frame
sv = solar_vector_moon_frame(et_from_iso(["2024-01-01T00:00:00"]))
assert abs(np.linalg.norm(sv[0]) - 1.0) < 1e-6, "should be a unit vector"
print("  SPICE kernels OK")

  SPICE kernels OK


## 1.5 Optional download
The Diviner surface temperatures (PDS, about 310 MB) are too large for the
repository. They are needed only to **recompute** the Diviner comparison (SI
Table S13); everything else works without them. Set `DOWNLOAD = True` and run
the cell to fetch them.

In [6]:
DOWNLOAD = False

DIV = [nb.CODE / f for f in DIVINER]
print(f"  Diviner tiles : {'present' if all(f.exists() for f in DIV) else 'not downloaded'}")
if DOWNLOAD and not all(f.exists() for f in DIV):
    nb.run("fetch_diviner.py")

  Diviner tiles : present


---
# 2. The analysis

Every step of the study, in the order in which the steps need each other (the
same order as `make analysis`). What runs depends on `MODE`; a step that
does not run says which stored result it uses. Notebooks 2 and 3 then read the
results.

## 2.1 The data and the earlier, fixed-albedo analyses
`retrieve_kd.py` is the study's first stage (the albedo held fixed). The later
steps reuse its sensor handling and outputs, and the fixed-albedo results
appear in Section 3.2 of the letter. `compute_stability_windows.py` lists every
sensor's stability window (SI Table S1), and `compute_insolation_check.py`
compares the real 1972–1976 sunlight with the idealized cycle (Section 2.2).

In [7]:
nb.step("compute/retrieve_kd.py", minutes=95, writes="kd_retrieval_results.json")
nb.step("compute/compute_common_epoch.py", minutes=15, writes="common_epoch_sensitivity.json")
nb.step("compute/compute_diviner_closure.py", minutes=3, writes="diviner_closure.json", needs=DIVINER)
nb.step("compute/compute_review_diagnostics.py", minutes=5, writes="review_diagnostics.json", needs=DIVINER)
nb.step("compute/compute_albedo_diagnostics.py", minutes=13, writes="albedo_diagnostics.json")
nb.step("compute/compute_albedo_sensitivity.py", minutes=35, writes="albedo_sensitivity.json")
nb.step("compute/compute_albedo_anchor.py", minutes=5, writes="albedo_anchor.json")
nb.step("compute/compute_stability_windows.py", minutes=0.2, writes="stability_windows.json")
nb.step("compute/compute_insolation_check.py", minutes=1, writes="insolation_check.json")

■ retrieve_kd.py: stored result used, results/kd_retrieval_results.json (2026-10-05). A full run takes about 1.6 h.
■ compute_common_epoch.py: stored result used, results/common_epoch_sensitivity.json (2026-10-05). A full run takes about 15 min.
■ compute_diviner_closure.py: stored result used, results/diviner_closure.json (2026-10-05). A full run takes about 3 min.
■ compute_review_diagnostics.py: stored result used, results/review_diagnostics.json (2026-10-05). A full run takes about 5 min.
■ compute_albedo_diagnostics.py: stored result used, results/albedo_diagnostics.json (2026-10-05). A full run takes about 13 min.
■ compute_albedo_sensitivity.py: stored result used, results/albedo_sensitivity.json (2026-10-05). A full run takes about 35 min.
■ compute_albedo_anchor.py: stored result used, results/albedo_anchor.json (2026-10-05). A full run takes about 5 min.
■ compute_stability_windows.py: stored result used, results/stability_windows.json (2026-10-05). A full run takes seconds.


## 2.2 The joint fit of the albedo and $K_d$
The main result (letter Sections 2.3, 3.3). At each site the model is solved
for 25 albedos × 94 values of $K_d$ and both are fitted to the sensor
temperatures, the mean surface temperature and the annual-wave diffusivity.
A full run takes about 25 min; "quick" re-analyses the stored grid of model
runs (`joint_albedo_fit_cache.npz`) in seconds, with the same minimum and the
same 1500-draw bootstrap. Then: each probe and each thermometer type on its
own (Figure 3), the solver tests at the fit and the check against the C++
copy of the solver (SI Table S3; it compiles `code/cpp/solver.cpp` if needed), the
worked example of the trade-off (SI Table S10), each probe's own diffusivity
and the ±5 K Apollo test (SI Table S9), and the bootstrap with the sensors
grouped by probe.

In [8]:
nb.step("compute/compute_joint_albedo_fit.py", minutes=25, writes="joint_albedo_fit.json",
        quick=("--reuse",), quick_needs="results/joint_albedo_fit_cache.npz")
nb.step("compute/certify_joint_numerics.py", minutes=3, writes="joint_numerics_certification.json")
nb.step("compute/benchmark_cpp.py", minutes=2, writes="cpp_benchmark.json")
nb.step("compute/compute_subset_fits.py", minutes=1, writes="subset_fits.json")
nb.step("compute/compute_joint_valley.py", minutes=0.2, writes="joint_valley.json")
nb.step("compute/compute_joint_probe_checks.py", minutes=0.3, writes="joint_probe_checks.json")
nb.step("compute/compute_joint_block_bootstrap.py", minutes=1, writes="joint_block_bootstrap.json")

■ compute_joint_albedo_fit.py: stored result used, results/joint_albedo_fit.json (2026-10-05). A full run takes about 25 min.
■ certify_joint_numerics.py: stored result used, results/joint_numerics_certification.json (2026-10-15). A full run takes about 3 min.
■ benchmark_cpp.py: stored result used, results/cpp_benchmark.json (2026-10-25). A full run takes about 2 min.
■ compute_subset_fits.py: stored result used, results/subset_fits.json (2026-10-24). A full run takes about 1 min.
■ compute_joint_valley.py: stored result used, results/joint_valley.json (2026-10-05). A full run takes seconds.
■ compute_joint_probe_checks.py: stored result used, results/joint_probe_checks.json (2026-10-12). A full run takes seconds.
■ compute_joint_block_bootstrap.py: stored result used, results/joint_block_bootstrap.json (2026-10-05). A full run takes about 1 min.


## 2.3 Uncertainty and tests
The error budget (SI Table S5; about 80 min, or the fast form when
`results/joint_fit_sensitivities_cache.npz` is on your computer: it is not on
GitHub, 27 MB), the model at the fit against each sensor with the probe and
thermometer-type splits (Table 1, Figure 3e, j), the Martinez & Siegler model
with its density fitted (about 20 min) and the density it would need for the
measured diffusivities (Section 4.1), the Diviner comparison, the
measured gradients, the radiative coefficient and the density (SI Table S14;
about 50 min) and the likelihood-ratio tests.

In [9]:
nb.step("compute/compute_joint_fit_sensitivities.py", minutes=80, writes="joint_fit_sensitivities.json",
        quick=("--reuse",), quick_needs="results/joint_fit_sensitivities_cache.npz")
nb.step("compute/compute_joint_fit_checks.py", minutes=4, writes="joint_fit_checks.json")
nb.step("compute/compute_martinez_density.py", minutes=20, writes="martinez_density_fit.json")
nb.step("compute/compute_martinez_diffusivity.py", minutes=2, writes="martinez_diffusivity.json")
nb.step("compute/compute_joint_diviner.py", minutes=3, writes="joint_diviner.json", needs=DIVINER)
nb.step("compute/compute_gradient_epochs.py", minutes=0.2, writes="gradient_epochs.json")
nb.step("compute/compute_joint_chi_density.py", minutes=50, writes="joint_chi_density.json")
nb.step("compute/compute_likelihood_ratio.py", minutes=0.2, writes="likelihood_ratio.json")

■ compute_joint_fit_sensitivities.py: stored result used, results/joint_fit_sensitivities.json (2026-10-13). A full run takes about 80 min.
■ compute_joint_fit_checks.py: stored result used, results/joint_fit_checks.json (2026-10-23). A full run takes about 4 min.
■ compute_martinez_density.py: stored result used, results/martinez_density_fit.json (2026-10-16). A full run takes about 20 min.
■ compute_martinez_diffusivity.py: stored result used, results/martinez_diffusivity.json (2026-10-25). A full run takes about 2 min.
■ compute_joint_diviner.py: stored result used, results/joint_diviner.json (2026-10-20). A full run takes about 3 min.
■ compute_gradient_epochs.py: stored result used, results/gradient_epochs.json (2026-10-05). A full run takes seconds.
■ compute_joint_chi_density.py: stored result used, results/joint_chi_density.json (2026-10-06). A full run takes about 50 min.
■ compute_likelihood_ratio.py: stored result used, results/likelihood_ratio.json (2026-10-23). A full run 

## 2.4 The annual wave, the warming after deployment, and the posterior
The annual wave measured in the full record and modeled with the real solar
forcing (SI Figure S5, SI Table S11), the joint fit with our own diffusivities, a
different near-surface layer against the annual wave, the darkening of the
surface at deployment (SI Table S8), and the posterior with the basal heat flow
and the density also free (SI Figure S6, SI Table S6). The posterior solves a grid
of models first (about 2 h, resumable); "quick" samples the stored grid
(`joint_mcmc_grid.npz`) in about a minute.

In [10]:
nb.step("compute/compute_annual_wave.py", minutes=3, writes="annual_wave.json")
nb.step("compute/compute_own_diffusivity_fit.py", minutes=0.3, writes="own_diffusivity_fit.json")
nb.step("compute/compute_annual_wave_surface_test.py", minutes=6, writes="annual_wave_surface_test.json")
nb.step("compute/compute_transient_warming.py", minutes=2, writes="transient_warming.json")
nb.step("compute/compute_joint_mcmc.py", minutes=120, writes="joint_mcmc.json",
        quick=("--sample",), quick_needs="results/joint_mcmc_grid.npz")

■ compute_annual_wave.py: stored result used, results/annual_wave.json (2026-10-06). A full run takes about 3 min.
■ compute_own_diffusivity_fit.py: stored result used, results/own_diffusivity_fit.json (2026-10-12). A full run takes seconds.
■ compute_annual_wave_surface_test.py: stored result used, results/annual_wave_surface_test.json (2026-10-12). A full run takes about 6 min.
■ compute_transient_warming.py: stored result used, results/transient_warming.json (2026-10-07). A full run takes about 2 min.
■ compute_joint_mcmc.py: stored result used, results/joint_mcmc.json (2026-10-05). A full run takes about 2.0 h.


---
## Next

The results are now in `code/results/`. `2_letter_results.ipynb` checks every
number of the letter against them and draws Figures 3 and 4;
`3_supporting_information.ipynb` does the same for the SI tables and draws SI
Figures S5–S7. The other figures of the paper illustrate the data and the
method (Figures 1 and 2, SI Figures S1–S4); `make figures` draws them.